## Xarray engine: auxiliary coordinates

In [1]:
import earthkit.data as ekd

### Basic examples

In [2]:
ds_fl = ekd.from_source("sample", "pl.grib").to_fieldlist()
ds_fl.ls().head()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2024-06-03 00:00:00,2024-06-03,0 days 00:00:00,700,pressure,0,regular_ll
1,r,2024-06-03 00:00:00,2024-06-03,0 days 00:00:00,700,pressure,0,regular_ll
2,t,2024-06-03 00:00:00,2024-06-03,0 days 00:00:00,500,pressure,0,regular_ll
3,r,2024-06-03 00:00:00,2024-06-03,0 days 00:00:00,500,pressure,0,regular_ll
4,t,2024-06-03 06:00:00,2024-06-03,0 days 06:00:00,700,pressure,0,regular_ll


In [3]:
ds = ds_fl.to_xarray(
    profile="grib",
    aux_coords={"expver": ("metadata.expver", "forecast_reference_time")},
)
ds.load()

<xarray.Dataset> Size: 176kB
Dimensions:                  (forecast_reference_time: 4, step: 2, level: 2,
                              latitude: 19, longitude: 36)
Coordinates:
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 32B 202...
    expver                   (forecast_reference_time) <U4 64B '0001' ... '0001'
  * step                     (step) timedelta64[ns] 16B 00:00:00 06:00:00
  * level                    (level) int64 16B 500 700
  * latitude                 (latitude) float64 152B 90.0 80.0 ... -80.0 -90.0
  * longitude                (longitude) float64 288B 0.0 10.0 ... 340.0 350.0
Data variables:
    r                        (forecast_reference_time, step, level, latitude, longitude) float64 88kB ...
    t                        (forecast_reference_time, step, level, latitude, longitude) float64 88kB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

In [4]:
ds2 = ds_fl.to_xarray(
    profile="grib",
    remapping={"centre_expver": "{metadata.centre}_{metadata.expver}"},
    aux_coords={"centre_and_expver": ("centre_expver", ("forecast_reference_time", "step"))},
)
ds2.load()

<xarray.Dataset> Size: 176kB
Dimensions:                  (forecast_reference_time: 4, step: 2, level: 2,
                              latitude: 19, longitude: 36)
Coordinates:
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 32B 202...
  * step                     (step) timedelta64[ns] 16B 00:00:00 06:00:00
    centre_and_expver        (forecast_reference_time, step) <U9 288B 'ecmf_0...
  * level                    (level) int64 16B 500 700
  * latitude                 (latitude) float64 152B 90.0 80.0 ... -80.0 -90.0
  * longitude                (longitude) float64 288B 0.0 10.0 ... 340.0 350.0
Data variables:
    r                        (forecast_reference_time, step, level, latitude, longitude) float64 88kB ...
    t                        (forecast_reference_time, step, level, latitude, longitude) float64 88kB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

The feature of declaring auxiliary coordinates works also with the mono_variable setting

In [5]:
ds3 = ds_fl.to_xarray(
    profile="grib",
    fixed_dims=["parameter.variable", "time.forecast_reference_time", "time.step", "vertical.level"],
    mono_variable=True,
    remapping={"centre_expver": "{metadata.centre}_{metadata.expver}"},
    aux_coords={"centre_and_expver": ("centre_expver", ("time.forecast_reference_time", "time.step"))},
)
ds3.load()

<xarray.Dataset> Size: 176kB
Dimensions:                  (variable: 2, forecast_reference_time: 4, step: 2,
                              level: 2, latitude: 19, longitude: 36)
Coordinates:
  * variable                 (variable) <U1 8B 'r' 't'
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 32B 202...
  * step                     (step) timedelta64[ns] 16B 00:00:00 06:00:00
    centre_and_expver        (forecast_reference_time, step) <U9 288B 'ecmf_0...
  * level                    (level) int64 16B 500 700
  * latitude                 (latitude) float64 152B 90.0 80.0 ... -80.0 -90.0
  * longitude                (longitude) float64 288B 0.0 10.0 ... 340.0 350.0
Data variables:
    data                     (variable, forecast_reference_time, step, level, latitude, longitude) float64 175kB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

### Quantiles in a probabilistic forecast

Let us now consider a probabilistic forecast of 2-metre temperature.

In [6]:
ds_fl2 = ekd.from_source("sample", "quantiles_pd.grib").to_fieldlist()

In this dataset, the fields are indexed by the metadata key ``"quantile"``, which is in turn composed of ``"number"`` and ``"numberOfForecastsInEnsemble"``

In [7]:
ds_fl2.ls(
    keys=[
        "metadata.shortName",
        "metadata.dataDate",
        "metadata.dataTime",
        "metadata.stepRange",
        "metadata.dataType",
        "metadata.quantile",
        "metadata.number",
        "metadata.numberOfForecastsInEnsemble",
    ]
)

,metadata.shortName,metadata.dataDate,metadata.dataTime,metadata.stepRange,metadata.dataType,metadata.quantile,metadata.number,metadata.numberOfForecastsInEnsemble
0,2tp,20251209,0,0-168,pd,1:3,1,3
1,2tp,20251209,0,0-168,pd,1:5,1,5
2,2tp,20251209,0,0-168,pd,1:10,1,10
3,2tp,20251209,0,0-168,pd,2:3,2,3
4,2tp,20251209,0,0-168,pd,2:5,2,5
5,2tp,20251209,0,0-168,pd,2:10,2,10
6,2tp,20251209,0,0-168,pd,3:3,3,3
7,2tp,20251209,0,0-168,pd,3:5,3,5
8,2tp,20251209,0,0-168,pd,3:10,3,10
9,2tp,20251209,0,0-168,pd,4:5,4,5


Note that, in this context, the usual meaning of the metadata key ``"number"`` (and the related ``"numberOfForecastsInEnsemble"``) is overridden by ``"quantile"``. As a result, the ensemble dimension normally derived from ``"number"`` is no longer applicable.

For this reason, we must:
- declare ``"quantile"`` as an extra dimension, and
- remove the predefined ensemble dimension ``"number"``, since it would otherwise conflict with the ``"quantile"`` dimension.

Still, it might be useful to keep the information carried by ``"number"`` and ``"numberOfForecastsInEnsemble"`` is auxiliary coordinates.

In [8]:
ds4 = ds_fl2.to_xarray(
    profile="grib",
    squeeze=False,
    extra_dims="metadata.quantile",
    drop_dims="member",
    add_earthkit_attrs=False,
    aux_coords={
        "quantile_rank": ("ensemble.member", "metadata.quantile"),
        "nquantiles": ("metadata.numberOfForecastsInEnsemble", "metadata.quantile"),
    },
)
ds4.load()

<xarray.Dataset> Size: 13kB
Dimensions:                  (quantile: 18, forecast_reference_time: 1,
                              step: 1, level: 1, level_type: 1, latitude: 7,
                              longitude: 12)
Coordinates:
  * quantile                 (quantile) <U5 360B '10:10' '1:10' ... '9:10'
    quantile_rank            (quantile) <U2 144B '10' '1' '1' ... '7' '8' '9'
    nquantiles               (quantile) int64 144B 10 10 3 5 10 ... 10 10 10 10
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 8B 2025...
  * step                     (step) timedelta64[ns] 8B 7 days
  * level                    (level) int64 8B 0
  * level_type               (level_type) <U7 28B 'surface'
  * latitude                 (latitude) float64 56B 90.0 60.0 ... -60.0 -90.0
  * longitude                (longitude) float64 96B 0.0 30.0 ... 300.0 330.0
Data variables:
    2tp                      (quantile, forecast_reference_time, step, level, level_type, latitude, longitude) float64 12kB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

### Monthly statistics

In [9]:
ds_fl3 = ekd.from_source("file", "/Users/ecm8620/data/issue-948-avg_2t-2months.grib2").to_fieldlist()
ds_fl3.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,avg_2t,2010-07-01,2010-06-01,30 days,2,height_above_ground_level,0,regular_ll
1,avg_2t,2010-08-01,2010-07-01,31 days,2,height_above_ground_level,0,regular_ll


In [10]:
ds5 = ds_fl3.to_xarray(
    drop_dims="step",
    aux_coords={"step": ("time.step", ("forecast_reference_time",))},
)
ds5.load()

<xarray.Dataset> Size: 22kB
Dimensions:                  (forecast_reference_time: 2, latitude: 33,
                              longitude: 41)
Coordinates:
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 16B 201...
    step                     (forecast_reference_time) timedelta64[ns] 16B 30...
  * latitude                 (latitude) float64 264B 55.0 54.75 ... 47.25 47.0
  * longitude                (longitude) float64 328B 5.0 5.25 ... 14.75 15.0
Data variables:
    avg_2t                   (forecast_reference_time, latitude, longitude) float64 22kB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

In [30]:
ds5 = ds_fl3.to_xarray(
    time_dim_mode="valid_time",
    extra_dims=[{"s": "metadata.expver"}],
    ensure_dims="s",
    # aux_coords={"step": ("time.step", ("forecast_reference_time", ))},
    # aux_coords={"time2": ("time.forecast_reference_time", "valid_time")}, # ISSUE !!!!
)
ds5.load()

<xarray.Dataset> Size: 22kB
Dimensions:     (s: 1, valid_time: 2, latitude: 33, longitude: 41)
Coordinates:
  * s           (s) <U4 16B '0001'
  * valid_time  (valid_time) datetime64[ns] 16B 2010-07-01 2010-08-01
  * latitude    (latitude) float64 264B 55.0 54.75 54.5 ... 47.5 47.25 47.0
  * longitude   (longitude) float64 328B 5.0 5.25 5.5 5.75 ... 14.5 14.75 15.0
Data variables:
    avg_2t      (s, valid_time, latitude, longitude) float64 22kB 284.2 ... 2...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF